# Results
- Evaluate models on GPU.

In [1]:
import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from eval_gpu import evaluate
from SReT import SReT_T_distill
import PiT_ToMe
import SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128
dataset_dir = '/media/datasets/imagenet/val'
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


#### DeiT-Tiny-Distill 

In [2]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.40 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          2.17 G
Throughput (BS=128):            1799.16 images/sec
Throughput (BS=64):             1923.67 images/sec
Throughput (BS=32):             2003.98 images/sec
Throughput (BS=16):             2329.31 images/sec
Throughput (BS=1):               742.02 images/sec
Peak Activation Memory (BS=128):         227.20 MB
Peak Activation Memory (BS=64):          113.03 MB
Peak Activation Memory (BS=32):           56.44 MB
Peak Activation Memory (BS=16):           28.59 MB
Peak Activation Memory (BS=1):             1.76 MB



#### DeiT-Tiny-Distill + ToMe

In [3]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()

rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model.r = r
    _ = evaluate(model, dataset_loader)

--- r = 0 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.40 %
Total Parameters:                           5.91 M
Theoretical FLOPs:                          2.17 G
Throughput (BS=128):            1787.25 images/sec
Throughput (BS=64):             1923.05 images/sec
Throughput (BS=32):             1987.09 images/sec
Throughput (BS=16):             2421.92 images/sec
Throughput (BS=1):               683.68 images/sec
Peak Activation Memory (BS=128):         247.01 MB
Peak Activation Memory (BS=64):          125.26 MB
Peak Activation Memory (BS=32):           61.83 MB
Peak Activation Memory (BS=16):           31.24 MB
Peak Activation Memory (BS=1):             1.93 MB

--- r = 10 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.47 %
Total Parameters:                           5.91 M
T

#### PiT-Tiny-Distill

In [4]:
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.42 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          1.01 G
Throughput (BS=128):            1794.38 images/sec
Throughput (BS=64):             1841.43 images/sec
Throughput (BS=32):             1930.08 images/sec
Throughput (BS=16):             2059.85 images/sec
Throughput (BS=1):               655.85 images/sec
Peak Activation Memory (BS=128):        1203.36 MB
Peak Activation Memory (BS=64):          603.73 MB
Peak Activation Memory (BS=32):          302.11 MB
Peak Activation Memory (BS=16):          151.92 MB
Peak Activation Memory (BS=1):             9.40 MB



#### PiT + ToMe (Constant Reduction)

In [2]:
rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model = PiT_ToMe.pit_ti_distilled(pretrained=True, constant_r=r)
    model = model.cuda().eval()
    _ = evaluate(model, dataset_loader)

--- r = 0 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            74.42 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          1.01 G
Throughput (BS=128):            1514.84 images/sec
Throughput (BS=64):             1553.18 images/sec
Throughput (BS=32):             1635.70 images/sec
Throughput (BS=16):             1741.24 images/sec
Throughput (BS=1):               546.53 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.44 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB

--- r = 10 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            73.76 %
Total Parameters:                           5.10 M
T

#### PiT + ToMe (Dynamic Reduction)

In [3]:
model = PiT_ToMe.pit_ti_distilled(pretrained=True, initial_r_ratio=0.25, alpha=0.2)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.62 %
Total Parameters:                           5.10 M
Theoretical FLOPs:                          0.77 G
Throughput (BS=128):            1911.62 images/sec
Throughput (BS=64):             1995.79 images/sec
Throughput (BS=32):             2035.70 images/sec
Throughput (BS=16):             1977.64 images/sec
Throughput (BS=1):               280.98 images/sec
Peak Activation Memory (BS=128):        1193.10 MB
Peak Activation Memory (BS=64):          597.80 MB
Peak Activation Memory (BS=32):          299.43 MB
Peak Activation Memory (BS=16):          150.61 MB
Peak Activation Memory (BS=1):             9.32 MB



#### SReT-Tiny-Distill

In [5]:
model = SReT_T_distill(pretrained=False) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            77.42 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.91 G
Throughput (BS=128):            1066.60 images/sec
Throughput (BS=64):             1163.31 images/sec
Throughput (BS=32):             1245.12 images/sec
Throughput (BS=16):             1295.46 images/sec
Throughput (BS=1):               220.24 images/sec
Peak Activation Memory (BS=128):         795.76 MB
Peak Activation Memory (BS=64):          397.88 MB
Peak Activation Memory (BS=32):          200.88 MB
Peak Activation Memory (BS=16):          100.44 MB
Peak Activation Memory (BS=1):             6.22 MB



#### SReT + ToMe (Constant Reduction)

In [6]:
rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model = SReT_ToMe.SReT_T_distill(pretrained=False, constant_r=r) # initialize
    checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    model = model.cuda().eval()
    _ = evaluate(model, dataset_loader)

--- r = 0 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            77.39 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.91 G
Throughput (BS=128):             945.14 images/sec
Throughput (BS=64):             1026.53 images/sec
Throughput (BS=32):             1115.47 images/sec
Throughput (BS=16):             1154.93 images/sec
Throughput (BS=1):               191.91 images/sec
Peak Activation Memory (BS=128):         784.66 MB
Peak Activation Memory (BS=64):          392.33 MB
Peak Activation Memory (BS=32):          199.10 MB
Peak Activation Memory (BS=16):           99.05 MB
Peak Activation Memory (BS=1):             6.13 MB

--- r = 10 ---
Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            70.89 %
Total Parameters:                           4.76 M
T

#### SReT + ToMe (Dynamic Reduction)

In [ ]:
# r=0.25, a=0.2
model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.25, alpha=0.2) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            75.24 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.40 G
Throughput (BS=128):            1423.80 images/sec
Throughput (BS=64):             1559.07 images/sec
Throughput (BS=32):             1620.45 images/sec
Throughput (BS=16):             1604.04 images/sec
Throughput (BS=1):               143.80 images/sec
Peak Activation Memory (BS=128):         489.38 MB
Peak Activation Memory (BS=64):          245.46 MB
Peak Activation Memory (BS=32):          122.71 MB
Peak Activation Memory (BS=16):           61.58 MB
Peak Activation Memory (BS=1):             3.90 MB



In [5]:
# r=0.40, a=0.1
model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.40, alpha=0.1) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                            71.60 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          1.20 G
Throughput (BS=128):            1771.09 images/sec
Throughput (BS=64):             1914.95 images/sec
Throughput (BS=32):             1951.66 images/sec
Throughput (BS=16):             1885.03 images/sec
Throughput (BS=1):               149.13 images/sec
Peak Activation Memory (BS=128):         391.51 MB
Peak Activation Memory (BS=64):          195.76 MB
Peak Activation Memory (BS=32):           99.88 MB
Peak Activation Memory (BS=16):           48.94 MB
Peak Activation Memory (BS=1):             3.90 MB



In [6]:
# r=0.60, a=0.40
model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.60, alpha=0.4) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader)

Target Batch Size:                             128
--------------------------------------------------
Top-1 Accuracy:                             8.14 %
Total Parameters:                           4.76 M
Theoretical FLOPs:                          0.62 G
Throughput (BS=128):            3077.22 images/sec
Throughput (BS=64):             3110.54 images/sec
Throughput (BS=32):             2548.08 images/sec
Throughput (BS=16):             1749.50 images/sec
Throughput (BS=1):               129.28 images/sec
Peak Activation Memory (BS=128):         391.51 MB
Peak Activation Memory (BS=64):          195.76 MB
Peak Activation Memory (BS=32):           99.88 MB
Peak Activation Memory (BS=16):           48.94 MB
Peak Activation Memory (BS=1):             3.90 MB

